# Notebook 2 — Transmission Data: ECB Policy Cycles and Credit Heterogeneity

This notebook documents the ECB policy rate history, characterises heterogeneous
transmission across euro area countries using BLS data, estimates country-level
transmission sensitivities, and produces the hypothesis preview scatter plot
linking production network concentration to transmission strength.

**Run Notebook 1 first** to generate the country topology data.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr, linregress
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from transmission_data import (
    generate_synthetic_transmission_data,
    load_bls_from_csv,
    construct_transmission_panel,
    save_transmission_data,
)
from panel_regression import merge_network_and_transmission, BLS_TO_TIVA, BLS_REGRESSION_EXCLUDE
from network_metrics import load_metrics_panel

DATA_PROC = Path('../data/processed')
RESULTS   = Path('../results/figures')
TABLES    = Path('../results/tables')
for p in [DATA_PROC, RESULTS, TABLES]:
    p.mkdir(parents=True, exist_ok=True)

TIGHTENING_CYCLES = {
    '2000–2001': ('2000-02-01', '2000-10-31', '#FF6B35'),
    '2005–2007': ('2005-12-01', '2007-07-31', '#FF006E'),
    '2011':      ('2011-04-01', '2011-07-31', '#FFBE0B'),
    '2022–2023': ('2022-07-01', '2023-09-30', '#8338EC'),
}
print('Setup complete.')

---
## 2.1 Load Data

In [ ]:
bls_path  = DATA_PROC / 'bls_net_tightening.parquet'
rate_path = DATA_PROC / 'ecb_policy_rate.parquet'

if bls_path.exists() and rate_path.exists():
    bls_df = pd.read_parquet(bls_path)
    policy_rate = pd.read_parquet(rate_path).iloc[:, 0]
    print(f'BLS loaded: {bls_df.shape}')
else:
    print('BLS data not found — run Notebook 1 first.')
    raise FileNotFoundError(bls_path)

bls_df.index = pd.to_datetime(bls_df.index)
policy_rate.index = pd.to_datetime(policy_rate.index)
delta_rate = policy_rate.diff().rename('delta_rate')

topo_path = DATA_PROC / 'country_topology.parquet'
country_topo = pd.read_parquet(topo_path) if topo_path.exists() else None

print(f'Policy rate: {policy_rate.min():.2f}% – {policy_rate.max():.2f}%')
print(f'BLS countries: {sorted(bls_df.columns.tolist())}')
print(f'Topology: {len(country_topo) if country_topo is not None else "not found"} obs')

---
## 2.2 ECB Policy Rate History

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
ax.tick_params(colors='#c9d1d9')
for sp in ax.spines.values(): sp.set_color('#30363d')

for label, (start, end, color) in TIGHTENING_CYCLES.items():
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    if s >= policy_rate.index.min():
        ax.axvspan(s, e, alpha=0.18, color=color)
        ax.text(s + (e-s)/2, policy_rate.max()*0.88, label,
                ha='center', color=color, fontsize=8, fontweight='bold')

ax.plot(policy_rate.index, policy_rate.values, color='#FF6B35', lw=2.5)
ax.fill_between(policy_rate.index, policy_rate.values, alpha=0.15, color='#FF6B35')
ax.axhline(0, color='white', ls='--', lw=0.8, alpha=0.4)
ax.set_ylabel('ECB MRO rate (%)', color='#c9d1d9')
ax.set_title('ECB Policy Rate — Four Tightening Cycles (shaded)',
             color='white', fontsize=12)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
plt.setp(ax.xaxis.get_majorticklabels(), color='#c9d1d9')
plt.tight_layout()
plt.savefig(RESULTS / 'ecb_policy_rate.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('Tightening cycle magnitudes:')
for label, (start, end, _) in TIGHTENING_CYCLES.items():
    sub = policy_rate.loc[pd.Timestamp(start):pd.Timestamp(end)]
    if len(sub) >= 2:
        print(f'  {label}: {sub.iloc[-1] - sub.iloc[0]:+.2f}pp '
              f'over {len(sub)} quarters')

---
## 2.3 Heterogeneous Transmission — The Puzzle

Same ECB policy rate, different country responses. This heterogeneity motivates the production network explanation.

In [ ]:
DISPLAY = {
    'DE': ('#FF6B35', 'Germany'),    'FR': ('#4CC9F0', 'France'),
    'IT': ('#7B2FBE', 'Italy'),      'ES': ('#FFBE0B', 'Spain'),
    'NL': ('#FF006E', 'Netherlands'),'PT': ('#06D6A0', 'Portugal'),
}
avail = {k: v for k, v in DISPLAY.items() if k in bls_df.columns}

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#c9d1d9')
    for sp in ax.spines.values(): sp.set_color('#30363d')
    for _, (s, e, c) in TIGHTENING_CYCLES.items():
        if pd.Timestamp(s) >= bls_df.index.min():
            ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.10, color=c)
    ax.axhline(0, color='white', ls='--', lw=0.8, alpha=0.4)

for code, (color, name) in avail.items():
    axes[0].plot(bls_df.index, bls_df[code], color=color, lw=2, label=name)
axes[0].set_ylabel('Net tightening (%)', color='#c9d1d9')
axes[0].set_title('BLS Net Credit Tightening — Same ECB Rate, Different Country Responses',
                   color='white', fontsize=12)
axes[0].legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=9, ncol=3)

cycle = bls_df.loc['2022-01-01':'2024-06-30']
ax2 = axes[1].twinx()
ax2.plot(policy_rate.loc['2022-01-01':'2024-06-30'].index,
         policy_rate.loc['2022-01-01':'2024-06-30'].values,
         color='white', lw=1.5, ls=':', alpha=0.5, label='ECB rate (right)')
ax2.set_ylabel('ECB rate (%)', color='#8b949e', fontsize=9)
ax2.tick_params(colors='#8b949e')
for code, (color, name) in avail.items():
    if code in cycle.columns:
        axes[1].plot(cycle.index, cycle[code], color=color, lw=2.5, label=name)
axes[1].set_ylabel('Net tightening (%)', color='#c9d1d9')
axes[1].set_title('2022–2023 Tightening Cycle', color='white', fontsize=11)
axes[1].legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=9, ncol=3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=45, ha='right', color='#c9d1d9')
plt.tight_layout()
plt.savefig(RESULTS / 'bls_heterogeneity.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('Mean tightening 2022Q3–2023Q3:')
for code, (_, name) in avail.items():
    mean = bls_df.loc['2022-07-01':'2023-09-30', code].mean()
    print(f'  {name:<16}: {mean:+.1f}%')

---
## 2.4 Country Transmission Sensitivities

Univariate regression of BLS on ECB rate changes for each country. Cyprus, Malta, and Luxembourg are excluded (see panel_regression.py for rationale).

In [ ]:
EXCLUDE = BLS_REGRESSION_EXCLUDE | {'U2'}

sensitivities = {}
for code in bls_df.columns:
    if code in EXCLUDE:
        continue
    s_bls  = bls_df[code].dropna()
    s_rate = delta_rate.reindex(s_bls.index).dropna()
    common = s_bls.index.intersection(s_rate.index)
    if len(common) < 20:
        continue
    slope, _, r, p, se = linregress(s_rate.loc[common], s_bls.loc[common])
    sensitivities[code] = {'sensitivity': slope, 'r2': r**2, 'p': p, 'se': se}

sens_df = (pd.DataFrame(sensitivities).T
             .sort_values('sensitivity', ascending=False))

print('Transmission sensitivities (BLS pp per 100bps ECB hike):')
print(sens_df.round(3).to_string())
sens_df.to_csv(TABLES / 'transmission_sensitivities.csv')

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
ax.tick_params(colors='#c9d1d9')
for sp in ax.spines.values(): sp.set_color('#30363d')

values  = sens_df['sensitivity'].values
colors_ = ['#FF006E' if v < sens_df['sensitivity'].median() else '#4CC9F0'
           for v in values]
ax.bar(sens_df.index, values, color=colors_, alpha=0.85, edgecolor='#30363d')
ax.axhline(sens_df['sensitivity'].median(), color='white', ls='--', lw=1.5,
           alpha=0.6, label=f'Median ({sens_df["sensitivity"].median():.2f})')
for i, (bar, pval) in enumerate(zip(ax.patches, sens_df['p'].values)):
    if pval < 0.10:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + (0.5 if bar.get_height() > 0 else -0.8),
                '*', ha='center', color='white', fontsize=12)
ax.set_ylabel('Sensitivity (BLS pp / 100bps)', color='#c9d1d9')
ax.set_title('Country Transmission Sensitivities
(* p<0.10; red = below median)',
             color='white', fontsize=11)
ax.legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS / 'transmission_sensitivities.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
## 2.5 Hypothesis Preview — NCI vs Transmission Sensitivity

Bivariate correlation between production network concentration and transmission sensitivity. The formal test with fixed effects follows in Notebook 4.

In [ ]:
if country_topo is not None and len(sens_df) >= 4:
    ct = country_topo.reset_index()
    latest = (ct[ct['year'] == ct['year'].max()]
              .set_index('country'))

    sens_df['tiva'] = sens_df.index.map(BLS_TO_TIVA)
    plot_df = (sens_df.dropna(subset=['tiva'])
               .join(latest[['nci', 'mean_upstreamness', 'centrality_hhi']],
                     on='tiva', how='inner'))

    if len(plot_df) >= 4:
        fig, axes = plt.subplots(1, 2, figsize=(13, 6))
        fig.patch.set_facecolor('#0d1117')
        for ax in axes:
            ax.set_facecolor('#161b22')
            ax.tick_params(colors='#c9d1d9')
            for sp in ax.spines.values(): sp.set_color('#30363d')

        for ax, xcol, xlabel in [
            (axes[0], 'nci',              'Network Centralisation Index (NCI)'),
            (axes[1], 'mean_upstreamness','Mean Upstreamness'),
        ]:
            x = plot_df[xcol].values
            y = plot_df['sensitivity'].values
            ax.scatter(x, y, color='#4CC9F0', s=90, alpha=0.85,
                       edgecolors='#30363d', zorder=3)
            for idx, row in plot_df.iterrows():
                ax.annotate(str(idx), (row[xcol], row['sensitivity']),
                            xytext=(5, 3), textcoords='offset points',
                            color='#c9d1d9', fontsize=9)
            if len(x) >= 4:
                sl, ic, r, p, _ = linregress(x, y)
                xl = np.linspace(x.min(), x.max(), 50)
                ax.plot(xl, sl*xl+ic, color='#FF6B35', lw=2, ls='--',
                        label=f'β={sl:.2f}  r²={r**2:.3f}  p={p:.3f}')
            ax.set_xlabel(xlabel, color='#c9d1d9')
            ax.set_ylabel('Transmission sensitivity', color='#c9d1d9')
            ax.legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=8)

        axes[0].set_title('Hypothesis Preview: Higher NCI → Weaker Transmission?
'
                           '(bivariate; formal test with fixed effects in Notebook 4)',
                           color='white', fontsize=10)
        axes[1].set_title('Higher Upstreamness → Weaker Transmission?',
                           color='white', fontsize=10)
        plt.tight_layout()
        plt.savefig(RESULTS / 'hypothesis_preview.png', dpi=150,
                    bbox_inches='tight', facecolor='#0d1117')
        plt.show()

        print('Bivariate correlations (no controls):')
        for col in ['nci', 'mean_upstreamness', 'centrality_hhi']:
            if col in plot_df.columns:
                r, p = pearsonr(plot_df[col].fillna(0),
                                plot_df['sensitivity'].fillna(0))
                sig = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else ''
                print(f'  sensitivity ~ {col:<22}: r={r:.3f}  p={p:.3f} {sig}')
    else:
        print(f'Only {len(plot_df)} matched countries for scatter plot.')
else:
    print('Run Notebook 1 to generate country topology data.')

---
## 2.6 Build Merged Panel

In [ ]:
transmission_panel = construct_transmission_panel(bls_df, policy_rate)
print(f'Transmission panel: {len(transmission_panel):,} obs')
print(transmission_panel[['bls_tightening','delta_bls','ecb_rate',
                            'delta_rate']].describe().round(3))

if country_topo is not None:
    metrics_path = DATA_PROC / 'network_metrics_panel.parquet'
    if metrics_path.exists():
        metrics_panel = load_metrics_panel(DATA_PROC)
        merged = merge_network_and_transmission(
            metrics_panel, country_topo, transmission_panel
        )
        merged.to_parquet(DATA_PROC / 'merged_regression_panel.parquet')
        print(f'Merged panel: {len(merged):,} obs — saved for Notebook 4')
    else:
        print('Metrics panel not found. Run Notebook 1 first.')

transmission_panel.to_parquet(DATA_PROC / 'transmission_panel.parquet')
transmission_panel[['bls_tightening','delta_bls','ecb_rate','delta_rate']]    .describe().round(3).to_csv(TABLES / 'transmission_summary_stats.csv')

print('\n--- STEP 2 COMPLETE ---')